# 🚗 Proyecto Final de Máster en IA: Aprendizaje por Refuerzo y Conducción Autónoma

Bienvenidos al reto final de Aprendizaje por Refuerzo (Deep RL). Vuestro objetivo es entrenar un agente capaz de conducir un vehículo autónomo en el simulador 3D **Duckietown**, utilizando únicamente datos de píxeles en bruto (cámara del salpicadero).

Un agente que memoriza un circuito es inútil. Deberéis entrenar en múltiples mapas y vuestra nota final dependerá del rendimiento del agente en un **mapa oculto** con obstáculos estáticos.

### 📋 Fases del Proyecto

* **Fase 1: Calentamiento.** Implementar Q-Learning tabular desde cero en `FrozenLake-v1`.
* **Fase 2: Baselines (DQN y PPO).** Implementar dos baselines usando la librería Stable Baselines3. Dado que Duckietown usa acciones continuas, tendréis que crear un Wrapper para discretizarlas en DQN.
* **Fase 3: Algoritmo Avanzado.** Diseñar y ajustar un tercer algoritmo de libre elección (ej. SAC, TD3, A2C, o una versión fuertemente optimizada de PPO con Curriculum Learning) que supere a los baselines.

### ⚠️ EL CONTRATO DE EVALUACIÓN (LEER ATENTAMENTE)
La evaluación está automatizada. Si vuestro código no funciona a la primera en la máquina del profesor, la nota de código será 0.
1.  **Observaciones:** El modelo DEBE esperar un espacio de observación con forma `(1, 64, 64)` apilado en 4 frames (Frame Stacking).
2.  **Clases personalizadas:** Vuestro archivo de evaluación debe incluir las definiciones exactas de las clases de Python de vuestra CNN y Wrappers.
3.  **Dry-Run:** Antes de entregar, debéis ejecutar todo en un Google Colab limpio para verificar que las dependencias funcionan.


## Fase 1: Calentamiento Tabular (Q-Learning)
Demostrad que entendéis la Ecuación de Bellman. Implementad Q-Learning desde cero (sin librerías de Deep RL) para resolver el entorno `FrozenLake-v1` (cuadrícula 8x8, resbaladiza).


In [ ]:
import gymnasium as gym
import numpy as np
import matplotlib.pyplot as plt

def train_q_learning(episodes=10000):
    # TODO: Inicializar el entorno FrozenLake-v1 (8x8)
    # TODO: Crear la Q-Table (matriz de ceros)
    # TODO: Implementar el bucle de entrenamiento con la Ecuación de Bellman
    # TODO: Implementar decaimiento de epsilon (Epsilon-Greedy)
    # TODO: Guardar y devolver la recompensa acumulada por episodio
    pass

# Ejecución y visualización
# rewards = train_q_learning()
# plt.plot(np.convolve(rewards, np.ones(100)/100, mode='valid')) # Media móvil
# plt.title('Progreso Q-Learning')
# plt.show()

## Fases 2 y 3: Conducción Autónoma en Duckietown

En esta sección, construiréis el pipeline para DQN, PPO y vuestro **Algoritmo Avanzado (Fase 3)**.
Aquí tenéis el código base para inicializar la pantalla virtual que requiere Google Colab y el esqueleto de procesamiento de imágenes.

**Mapas de Entrenamiento Permitidos:**
`Duckietown-loop_empty-v0`, `Duckietown-udem1-v0`, `Duckietown-zigzag_dists-v0`, `Duckietown-small_loop-v0`, `Duckietown-straight_road-v0`.

In [ ]:
import os
import pyvirtualdisplay
import gym as old_gym
import gymnasium as gym
from gymnasium import spaces
import gym_duckietown
import cv2
import numpy as np
import torch
import torch.nn as nn
from stable_baselines3 import PPO, DQN
# TODO: Importar vuestro tercer algoritmo (ej. SAC, TD3, etc.)

# INICIAR PANTALLA VIRTUAL (Obligatorio en Colab)
display = pyvirtualdisplay.Display(visible=False, size=(800, 600))
display.start()

class DuckieWrapper(gym.Env):
    """
    Wrapper que adapta Duckietown a Gymnasium, recorta la imagen (quita el cielo),
    la pasa a escala de grises y la redimensiona a 64x64.
    """
    def __init__(self, env_name="Duckietown-loop_empty-v0"):
        super().__init__()
        self.env = old_gym.make(env_name)

        # TODO: Ajustar el Action Space si usáis DQN (discreto) o PPO/SAC (continuo)
        self.action_space = spaces.Box(low=np.array([-1.0, -1.0]), high=np.array([1.0, 1.0]), dtype=np.float32)
        self.observation_space = spaces.Box(low=0, high=255, shape=(1, 64, 64), dtype=np.uint8)

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        obs = self.env.reset()
        return self._process_obs(obs), {}

    def step(self, action):
        obs, reward, done, info = self.env.step(action)
        return self._process_obs(obs), reward, done, False, info

    def _process_obs(self, obs):
        # Recortar cielo, convertir a grises y redimensionar
        obs = obs[obs.shape[0]//2:, :, :]
        gray = cv2.cvtColor(obs, cv2.COLOR_RGB2GRAY)
        resized = cv2.resize(gray, (64, 64), interpolation=cv2.INTER_AREA)
        return np.expand_dims(resized, axis=0)

    def render(self): return self.env.render(mode='rgb_array')
    def close(self): self.env.close()

class CustomCNN(BaseFeaturesExtractor):
    """CNN Personalizada para extraer características de los píxeles."""
    def __init__(self, observation_space: gym.spaces.Box, features_dim: int = 256):
        super().__init__(observation_space, features_dim)
        # TODO: Diseñar vuestra propia red convolucional aquí
        pass

    def forward(self, observations: torch.Tensor) -> torch.Tensor:
        # TODO: Implementar el Forward pass
        pass

### Bucle de Entrenamiento
Entrenad vuestros modelos (DQN, PPO y el Algoritmo Avanzado). Recordad usar **Frame Stacking** (apilamiento de frames) para que el agente tenga percepción de la velocidad y giros. Guardad vuestro modelo final usando `model.save()`.

In [ ]:
from stable_baselines3.common.vec_env import DummyVecEnv, VecFrameStack

# 1. Definir entornos de entrenamiento (usad múltiples mapas)
# 2. Envolverlos en DummyVecEnv y VecFrameStack(n_stack=4)
# 3. Configurar policy_kwargs usando vuestra CustomCNN
# 4. Entrenar DQN
# 5. Entrenar PPO
# 6. Entrenar Algoritmo Fase 3
# 7. Guardar el mejor como "best_duckie_agent"

# TODO: Vuestro código de entrenamiento aquí

## Evaluación Visual y Generación de Video
Para que podáis ver a vuestro agente conducir, ejecutad esta celda. Creará un entorno de prueba, dejará que el agente conduzca y generará un vídeo MP4. **El profesor usará un código idéntico a este en el mapa oculto para puntuaros.**

In [ ]:
import imageio
from IPython.display import Video

# 1. Cargar el entorno de prueba
def make_test_env():
    return DuckieWrapper("Duckietown-small_loop-v0") # En la evaluación final esto será 'loop_obstacles'

test_env = DummyVecEnv([make_test_env])
test_env = VecFrameStack(test_env, n_stack=4)

# 2. Cargar vuestro modelo
# model = PPO.load("best_duckie_agent")

print("Evaluando agente...")
obs = test_env.reset()
frames = []

# 3. Bucle de conducción
for i in range(300):
    # Descomentar cuando tengáis el modelo
    # action, _ = model.predict(obs, deterministic=True)
    # obs, reward, done, info = test_env.step(action)

    # Simulación de acción aleatoria (Bórralo cuando uses tu modelo)
    action = [test_env.action_space.sample()]
    obs, reward, done, info = test_env.step(action)

    rgb_frame = test_env.envs[0].env.render(mode='rgb_array')
    frames.append(rgb_frame)

    if done[0]:
        print(f"Episodio finalizado en el paso {i}")
        break

test_env.close()

# 4. Guardar y mostrar video
video_path = 'duckie_eval_video.mp4'
imageio.mimsave(video_path, frames, fps=30)
Video(video_path, embed=True)

---

## 📦 Requisitos de Entrega y Proceso de Evaluación

Estimados alumnos,

Para asegurar que el proceso de corrección sea justo y ágil, debéis seguir estrictamente las siguientes directrices para vuestra entrega final.

### 1. Entregables Obligatorios

Vuestra entrega debe ser un único archivo comprimido que contenga exactamente lo siguiente:

* **El Código Fuente:** Vuestro cuaderno `.ipynb` completo (o, en su defecto, los scripts de Python `train.py` y `eval.ipynb`).
* **Dependencias:** Un archivo `requirements.txt` con las versiones de las librerías fijadas **estrictamente** usando `==` (por ejemplo, `stable-baselines3==2.2.1`). Esto es vital para evitar problemas de compatibilidad.
* **El Agente Entrenado:** El archivo comprimido de vuestro modelo final (debe llamarse `best_duckie_agent.zip`).
* **La Memoria Técnica:** Un documento `Report.pdf`. Aquí debéis justificar teórica y empíricamente la elección y configuración de vuestro tercer algoritmo (Fase 3) y comparar detalladamente su rendimiento frente a las baselines obligatorias de DQN y PPO.

### 2. Cómo seréis evaluados (El Contrato de Evaluación)

Quiero ser muy transparente sobre cómo probaré vuestros proyectos: **no voy a depurar vuestro código ni a re-entrenar vuestros modelos desde cero.**

El proceso de corrección será exactamente el siguiente:

1. Tomaré vuestro archivo `best_duckie_agent.zip`.
2. Lo cargaré en mi propio script de evaluación, ejecutándolo en un entorno limpio basado estrictamente en vuestro `requirements.txt`.
3. Ejecutaré la celda de evaluación, pero cambiando el mapa al escenario oculto que no habéis visto durante el entrenamiento: **`Duckietown-loop_obstacles-v0`**.

Si habéis entrenado el modelo respetando las reglas de forma (mismas dimensiones de entrada, variables definidas y clases guardadas correctamente en el código), vuestro agente se cargará a la primera. Veré la simulación de vuestro coche intentando sortear los obstáculos y seréis puntuados por su capacidad de generalización y supervivencia en este nuevo mapa.

**⚠️ Advertencia final:** Si al intentar cargar vuestro modelo el script falla por incompatibilidad de versiones, clases no encontradas o errores de dimensiones en los tensores (*shape errors*), **la calificación de la parte práctica será un 0**.

Os recomiendo encarecidamente que abráis un Google Colab completamente en blanco, instaléis solo lo que dice vuestro `requirements.txt`, subáis vuestro `.zip` e intentéis ejecutar la evaluación vosotros mismos antes de enviar el trabajo definitivo.

## ¡Mucho ánimo con el entrenamiento y buena suerte!

In [ ]:
!python --version

from IPython.display import clear_output
clear_output()
!sudo update-alternatives --install /usr/bin/python3 python3 /usr/bin/python3.11 1

# Choose one of the given alternatives:
!sudo update-alternatives --config python3
!python --version


There are 3 choices for the alternative python3 (providing /usr/bin/python3).

  Selection    Path                 Priority   Status
------------------------------------------------------------
  0            /usr/bin/python3.12   2         auto mode
  1            /usr/bin/python3.10   1         manual mode
* 2            /usr/bin/python3.11   1         manual mode
  3            /usr/bin/python3.12   2         manual mode

Press <enter> to keep the current choice[*], or type selection number: 2
Python 3.11.15
